# 03 — Market Basket Analysis
**Lumen Data Science — Cross-Sell & Up-Sell Prediction**

Cilj ovog notebooka:
1. Razumjeti teoriju — support, confidence, lift, antimonotonost
2. Pripremiti basket matricu na razini `Invoice #` × `Product group`
3. Pokrenuti **Apriori** i **FP-Growth** i usporediti ih
4. Generirati asocijacijska pravila filtrirana po lift-u
5. Izvući cross-sell preporuke po kupcu
6. Pripremiti output za ML model (notebook 05)

> **Napomena:** Up-sell logika (kupci koji kupuju ispod medijana cijene za PG)
> nije dio market basket analysis jer ne koristi co-occurrence podatke.
> To rješavamo u recommendation modelu / API-ju.

---
## Teorija — pojmovi koje trebamo znati

### Support
Koliko često se neka kombinacija proizvoda pojavljuje u transakcijama.
```
support({PG001, PG007}) = broj_faktura_s_oba / ukupno_faktura
```
Npr. 0.05 znači da se PG001 i PG007 zajedno nalaze na 5% svih faktura.

### Confidence
Koliko je vjerojatno da kupac kupi PG007 AKO već kupuje PG001.
```
confidence(PG001 → PG007) = support({PG001, PG007}) / support({PG001})
```

### Lift
Koliko je veza između PG001 i PG007 jača od slučajnosti.
```
lift(PG001 → PG007) = confidence(PG001 → PG007) / support({PG007})
```
- lift = 1 → nema veze, slučajnost
- lift > 1 → pozitivna asocijacija (ovo nam treba za cross-sell!)
- lift < 1 → negativna asocijacija (rijetko zajedno)

### Conviction
Mjeri koliko bi pravilo bilo krivo da su antecedent i konsekventa nezavisni.
```
conviction(PG001 → PG007) = (1 - support(PG007)) / (1 - confidence(PG001 → PG007))
```
- conviction = 1 → nezavisni
- conviction → ∞ → savršena asocijacija

### Antimonotonost (zašto Apriori funkcionira)
Ovo je ključna matematička osobina koja čini Apriori efikasnim:
> **Ako je {PG001, PG007} čest skup, onda {PG001} i {PG007} moraju biti česti.**
> Obrnuto: ako {PG001} nije čest, NIJEDAN skup koji sadrži PG001 ne može biti čest.

Apriori koristi ovo da 'reže grane' — ne gleda kombinacije čiji su podskupovi već rijetki.
FP-Growth radi isto ali bez generiranja kandidata — gradi stablo (FP-Tree) i traži obrasce direktno.
FP-Growth je znatno brži na velikim datasetima, Apriori je konceptualno jednostavniji.


In [ ]:
# ============================================================
# CELL 1: Setup i učitavanje
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import time

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

DATE_COLS = ['Invoice Date', 'Customer First Invoice Date']
df = pd.read_csv('../data/processed/data_clean_final.csv',
                 low_memory=False, parse_dates=DATE_COLS)

print(f'Shape: {df.shape}')
print(f'Stupci: {df.columns.tolist()}')

In [ ]:
# ============================================================
# CELL 2: EDA na razini Product group
# ============================================================
# Basket analysis radimo na razini Product group (ne Item Code)
# jer je Item Code pregranulan (~52k unikatnih) što bi dalo
# ekstremno rijetke kombinacije i gotovo nul support.
# Product group je prirodna razina za cross-sell preporuke.

print('=== Product group distribucija ===')
print(f'Unikatnih Product group: {df["Product group"].nunique()}')
print(f'Unikatnih Invoice #: {df["Invoice #"].nunique()}')
print(f'Unikatnih CustomerID: {df["CustomerID"].nunique()}')

pg_counts = df['Product group'].value_counts()
print(f'\nTop 15 Product group po broju redova:')
print(pg_counts.head(15))

# Vizualizacija
fig, ax = plt.subplots(figsize=(14, 5))
pg_counts.head(20).plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Top 20 Product Group po broju linija fakture')
ax.set_xlabel('Product Group')
ax.set_ylabel('Broj redova')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 3: Analiza složenosti faktura
# ============================================================
# Fakture s više product grupa su direktne cross-sell prilike.
# Fakture s 1 PG ne pridonose asocijacijskim pravilima.

pg_per_invoice = (
    df.groupby('Invoice #')['Product group']
    .nunique()
    .reset_index()
    .rename(columns={'Product group': 'n_pg'})
)

print('=== Distribucija broja Product group po fakturi ===')
print(pg_per_invoice['n_pg'].describe())

multi_pg = pg_per_invoice[pg_per_invoice['n_pg'] > 1]
single_pg = pg_per_invoice[pg_per_invoice['n_pg'] == 1]

print(f'\nFakture s 1 PG (ne koriste za basket): {len(single_pg):,} ({len(single_pg)/len(pg_per_invoice)*100:.1f}%)')
print(f'Fakture s 2+ PG (cross-sell prilike): {len(multi_pg):,} ({len(multi_pg)/len(pg_per_invoice)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(12, 4))
pg_per_invoice['n_pg'].clip(upper=15).value_counts().sort_index().plot(
    kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Broj unikatnih Product Group po fakturi (clip@15)')
ax.set_xlabel('Broj Product Group')
ax.set_ylabel('Broj faktura')
plt.tight_layout()
plt.show()

In [ ]:
# Vecina faktura (invoice #) je samo sa jednim proizvodom, to nam ne daje cross-sell pravila pa prelazimo na 
# kosarice u kojima su proizvodi kupljeni od jednog kupca u odredjenom mjesecu

In [ ]:
# Samo kupci s 2+ fakture (ostali nemaju razmak za računati)
repeat_customers = df.groupby('CustomerID')['Invoice #'].nunique()
repeat_ids = repeat_customers[repeat_customers >= 2].index

customer_gaps = (
    df[df['CustomerID'].isin(repeat_ids)]
    .sort_values(['CustomerID', 'Invoice Date'])
    .groupby('CustomerID')['Invoice Date']
    .apply(lambda x: x.diff().dt.days.median())
)

print(f'Kupaca s 2+ fakture: {len(repeat_ids):,} / {df["CustomerID"].nunique():,}')
print(customer_gaps.describe())
print(f'\nMedijan razmaka: {customer_gaps.median():.0f} dana')

In [ ]:
# ============================================================
# CELL 4: Priprema basket matrice
# ============================================================
# Basket = sve što kupac kupi unutar kalendarskog mjeseca.
#
# Zašto ne Invoice #?
# 98.2% faktura ima samo 1 Product group → gotovo nul co-occurrence.
# B2B nabavni ciklusi su mjesečni (medijan razmaka između kupnji = 20 dana),
# pa je CustomerID × mjesec prirodna "košarica".

# Korak 1: Definiraj basket kao CustomerID × godina-mjesec
df['basket_id'] = (
    df['CustomerID'].astype(str) + '_' +
    df['Invoice Date'].dt.to_period('M').astype(str)
)

basket_lists = (
    df.groupby('basket_id')['Product group']
    .apply(lambda x: list(x.dropna().unique()))
    .reset_index()
)
basket_lists.columns = ['basket_id', 'product_groups']

# Statistike — koliko je sada bogatije?
n_pg_per_basket = basket_lists['product_groups'].apply(len)
multi_pg = (n_pg_per_basket > 1).sum()
print(f'Ukupno basketa: {len(basket_lists):,}')
print(f'Basketa s 2+ PG: {multi_pg:,} ({multi_pg/len(basket_lists)*100:.1f}%)')
print(f'Prosječno PG po basketu: {n_pg_per_basket.mean():.2f}')
print(f'\nDistribucija PG po basketu:')
print(n_pg_per_basket.describe())

# Korak 2: TransactionEncoder → boolean matrica
transactions = basket_lists['product_groups'].tolist()

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_matrix = pd.DataFrame(te_array, columns=te.columns_)
basket_matrix = basket_matrix.drop(columns=['Unknown'], errors='ignore')

print(f'\nBasket matrica: {basket_matrix.shape}')
print(f'Stupci (Product Groups): {list(basket_matrix.columns)}')
print(f'\nPrimjer (prvih 5 redova, prvih 8 stupaca):')
display(basket_matrix.iloc[:5, :8])

In [ ]:
# ============================================================
# CELL 5: Sensitivity check za min_support
# ============================================================
# B2B dataset ima manje transakcija po kombinaciji nego retail.
# Moramo provjeriti koji min_support daje razuman broj pravila.

print('=== Sensitivity check: min_support vs. broj frequent itemsets ===\n')
print(f'{"Support":>10} | {"Itemsets":>10} | {"Parovi (2+)":>12} | {"Komentar"}')
print('-' * 65)

for s in [0.20, 0.15, 0.10, 0.05, 0.03, 0.02, 0.01]:
    try:
        fi = fpgrowth(basket_matrix, min_support=s, use_colnames=True)
        pairs = fi[fi['itemsets'].apply(len) >= 2]
        komentar = ''
        if len(pairs) == 0:
            komentar = '← premalo, nema parova'
        elif len(pairs) < 5:
            komentar = '← malo pravila'
        elif len(pairs) > 500:
            komentar = '← puno, može biti buke'
        else:
            komentar = '← dobro'
        print(f'{s:>10.2f} | {len(fi):>10,} | {len(pairs):>12,} | {komentar}')
    except Exception as e:
        print(f'{s:>10.2f} | ERROR: {e}')

print('\n→ Izaberi MIN_SUPPORT na temelju ovih rezultata.')

In [ ]:

# Provjera dal je ovo stvarno B2B dataset
ext = df

# 1. Koliko kupaca ukupno?
print(f'Unique CustomerIDs: {ext["CustomerID"].nunique():,}')
# B2B: tisuće do desetke tisuća. B2C retail: milijuni.

# 2. Revenue distribucija
cust_rev = ext.groupby('CustomerID').apply(
    lambda g: (g['Invoiced price'] * g['Invoiced qty (shipped)']).sum()
)
print(f'Median revenue/customer: ${cust_rev.median():,.0f}')
print(f'P90: ${cust_rev.quantile(0.90):,.0f}')
print(f'Max: ${cust_rev.max():,.0f}')

# 3. CustomerID format — B2B obično ima numeričke account IDs (kao tvoji),
# B2C webshopovi imaju email-based ili sekvencijalne user IDs s milijunima zapisa
print(f'Sample CustomerIDs: {ext["CustomerID"].unique()[:5]}')

# 4. Linije po kupcu
lines_per_cust = ext.groupby('CustomerID').size()
print(f'Median lines/customer: {lines_per_cust.median():.0f}')
print(f'P90 lines/customer: {lines_per_cust.quantile(0.90):.0f}')
# B2B: deseci do stotine. B2C: 1-5 obično.

In [ ]:
## Zašto je ovo B2B dataset

Dataset nije eksplicitno označen kao B2B, ali svi indikatori — strukturni i 
kvantitativni — to potvrđuju. Ova pretpostavka opravdava ključne metodološke 
odluke (customer-level basket, segmentaciju po industriji, tretman key accounta).

### Strukturni signali (schema)
- **Intercompany flag** — koncept međukompanijskih transfera postoji samo u 
  korporativnom kontekstu, ne u B2C maloprodaji.
- **Customer industry** (IC000, IC001...) — kupci klasificirani po industriji; 
  retail kupci nemaju "industriju".
- **Make vs Buy, Cost of part, Manufacturing Location Code** — manufacturing 
  vokabular; firma proizvodi/distribuira fizičke industrijske dijelove.
- **Top Customer Group, Customer First Invoice Date** — key account 
  management terminologija, isključivo B2B.
- **Ordered qty vs Invoiced qty (shipped)** — separacija narudžba/isporuka 
  je klasični B2B PO ciklus s djelomičnim isporukama.
- **CustomerID format** — šestznamenkasti numerički ERP account brojevi 
  (npr. 230494), tipični za SAP/Oracle/JD Edwards customer master file.

### Kvantitativni signali (mjereno na df_external, Intercompany=NO)
- **11,775 unique kupaca** — red veličine B2B-a (tisuće), ne B2C-a (milijuni).
- **Median revenue/customer = $6,737, P90 = $211,738, Max = $1.15B** — 
  ekstremna Pareto distribucija. Max/Median = ~171,000×. Milijardu dolara 
  po jednom kupcu je fizički nemoguće u B2C kontekstu.
- **Median 5 linija/customer, P90 = 87 linija** — bimodalna distribucija: 
  dugi rep malih povremenih kupaca + manjina velikih s tisućama linija. 
  Klasični B2B mix (mala lokalna firma kupuje 3 dijela godišnje vs. velika 
  korporacija s mjesečnim procurement ciklusima).
- **K-Means klasteri (iz notebook 02)** potvrđuju strukturu: 
  Whales (~1,760 kupaca, ~$2M avg revenue) = enterprise key accounti; 
  Active Small + Hibernating = dugi rep manjih B2B klijenata.

### Najvjerojatnija domena
B2B manufacturing ili industrijska distribucija fizičkih komponenti/dijelova. 
Manje vjerojatno: B2B distributer prema retail lancima (B2B2C). U oba slučaja 
metodologija ostaje ista — preporuke ciljaju business klijente, ne krajnje 
potrošače.

### Posljedice za modeliranje
1. **Customer-level basket** umjesto invoice-level — fakture u B2B-u su 
   PO-driven artefakti, ne reflektiraju "košarice komplementarnih proizvoda".
2. **Segmentacija po Customer industry** — različiti industrijski segmenti 
   imaju različite obrasce kupnje; agregiranje razrjeđuje signal.
3. **Cluster-conditional pravila** — pravila izračunata unutar K-Means 
   klastera relevantnija su od pravila iz cijele populacije.
4. **Up-sell logika unutar Product groupa** — kupci ispod medijana cijene 
   za PG su targeti za skuplje varijante.

In [ ]:
# ============================================================
# CELL 6: Apriori algoritam
# ============================================================
# Apriori je klasični algoritam (Agrawal & Srikant, 1994).
# Radi u više prolaza:
#   1. Nađe sve česte jednoelementne skupove (L1)
#   2. Iz L1 generira kandidate za parove (C2)
#   3. Provjeri C2, zadrži česte → L2
#   4. Ponavlja dok nema novih čestih skupova
#
# ANTIMONOTONOST u praksi: ako {PG001, PG007}
# nije čest, Apriori ne generira nijedan trojac koji ih sadrži.

# Prilagodi na temelju sensitivity checka!
MIN_SUPPORT = 0.01  # <-- promijeni ako treba

print(f'Pokrećem Apriori s min_support={MIN_SUPPORT}...')

t0 = time.time()
frequent_apriori = apriori(
    basket_matrix,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=3
)
t1 = time.time()

print(f'Apriori gotov za {t1-t0:.2f}s')
print(f'Ukupno frequent itemsets: {len(frequent_apriori)}')
print(f'  - jednočlani: {frequent_apriori[frequent_apriori["itemsets"].apply(len)==1].shape[0]}')
print(f'  - parovi: {frequent_apriori[frequent_apriori["itemsets"].apply(len)==2].shape[0]}')
print(f'  - trojci: {frequent_apriori[frequent_apriori["itemsets"].apply(len)==3].shape[0]}')

print('\nTop 15 frequent itemsets (parovi) po supportu:')
pairs_apriori = frequent_apriori[frequent_apriori['itemsets'].apply(len) == 2].sort_values(
    'support', ascending=False)
display(pairs_apriori.head(15))

In [ ]:
# ============================================================
# CELL 7: FP-Growth algoritam
# ============================================================
# FP-Growth (Han et al., 2000) ne generira kandidate eksplicitno.
# Gradi komprimiranu strukturu FP-Tree iz podataka,
# pa traži česte obrasce rekurzivno po stablu.
#
# Rezultati su IDENTIČNI kao Apriori za isti min_support.
# Razlika je samo u brzini — FP-Growth je brži.

print(f'Pokrećem FP-Growth s min_support={MIN_SUPPORT}...')

t0 = time.time()
frequent_fp = fpgrowth(
    basket_matrix,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=3
)
t1 = time.time()

print(f'FP-Growth gotov za {t1-t0:.2f}s')
print(f'Ukupno frequent itemsets: {len(frequent_fp)}')
print(f'  - jednočlani: {frequent_fp[frequent_fp["itemsets"].apply(len)==1].shape[0]}')
print(f'  - parovi: {frequent_fp[frequent_fp["itemsets"].apply(len)==2].shape[0]}')
print(f'  - trojci: {frequent_fp[frequent_fp["itemsets"].apply(len)==3].shape[0]}')

# Verifikacija
print('\nVerifikacija: Apriori == FP-Growth?')
# Sortiraj oba po itemsetu i usporedi
def normalize(df):
    return df.assign(
        itemsets_sorted=df['itemsets'].apply(lambda x: tuple(sorted(x)))
    ).sort_values('itemsets_sorted')[['itemsets_sorted', 'support']].reset_index(drop=True)

a = normalize(frequent_apriori)
f = normalize(frequent_fp)

if a.equals(f):
    print('✓ Apriori i FP-Growth daju IDENTIČNE rezultate (itemseti i supporti)')
else:
    print('✗ Razlika između algoritama — to ne bi smjelo biti, provjeri parametre')

# Koristimo FP-Growth rezultate dalje (brži, isti output)
frequent_itemsets = frequent_fp.copy()
print('\n→ Koristimo FP-Growth rezultate za daljnju analizu.')

In [ ]:
# ovdje smo samo gledali koliko dobijemo pravila za min lift 1 i min confidence 25%, ali nam trebaju jaca pravila pa 
# dizemo tu granicu
"""
# ============================================================
# CELL 8: Generiranje asocijacijskih pravila
# ============================================================
# Iz frequent itemsets generiramo pravila oblika A → B.
#
# Lift je primarna metrika jer confidence može biti visok
# slučajno (ako je B uvijek čest). Lift mjeri stvarnu
# asocijaciju iznad baseline-a.
#
# Filteri:
#   min_lift > 1.0 → samo pozitivne asocijacije
#   min_confidence > 0.2 → barem 20% vjerojatnost

MIN_LIFT = 1.0
MIN_CONFIDENCE = 0.25

rules_test = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=MIN_LIFT,
    num_itemsets=len(basket_matrix)
)

# Dodatni filter na confidence
rules_test = rules_test[rules_test['confidence'] >= MIN_CONFIDENCE].copy()

# Dodaj string verzije za lakše čitanje i CSV export
rules_test['antecedents_str'] = rules_test['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules_test['consequents_str'] = rules_test['consequents'].apply(lambda x: ', '.join(sorted(x)))

# Sortiraj po lift-u
rules_test = rules_test.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Ukupno pravila (lift>{MIN_LIFT}, confidence>{MIN_CONFIDENCE}): {len(rules_test)}')
print(f'\nTop 20 pravila po lift-u:')
display(rules_test[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'conviction']].head(30))
"""

In [ ]:
# isti cell kao i prije, ali malo cemo promjeniti min thresholde da dobijemo jaca pravila

MIN_LIFT = 2.0
MIN_CONFIDENCE = 0.30
MIN_CONVICTION = 1.3

rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=MIN_LIFT,
    num_itemsets=len(basket_matrix)
)

# Dodatni filter na confidence
rules = rules[
    (rules['confidence'] >= MIN_CONFIDENCE) &
    (rules['conviction'] >= MIN_CONVICTION)
].copy()

# Dodaj string verzije za lakše čitanje i CSV export
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))

# Sortiraj po lift-u
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Ukupno pravila (lift>{MIN_LIFT}, confidence>{MIN_CONFIDENCE}): {len(rules)}')
print(f'\nTop 20 pravila po lift-u:')
display(rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'conviction']])

In [ ]:
# ============================================================
# CELL 9: Analiza i vizualizacija pravila
# ============================================================

print('=== Statistike pravila ===')
print(rules[['support', 'confidence', 'lift', 'conviction', 'leverage']].describe().round(4))

# Scatter plot: support vs confidence, boja = lift
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    rules['support'],
    rules['confidence'],
    c=rules['lift'],
    cmap='YlOrRd',
    alpha=0.7,
    s=60
)
plt.colorbar(sc, ax=ax, label='Lift')
ax.set_xlabel('Support')
ax.set_ylabel('Confidence')
ax.set_title('Asocijacijska pravila: Support vs Confidence (boja = Lift)')
plt.tight_layout()
plt.show()

# Top 15 po lift-u — bar chart
top15 = rules.head(15).copy()
top15['rule'] = top15['antecedents_str'] + ' → ' + top15['consequents_str']

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(top15['rule'][::-1], top15['lift'][::-1], color='steelblue', edgecolor='black')
ax.axvline(x=1.0, color='red', linestyle='--', alpha=0.7, label='Lift=1 (slučajnost)')
ax.set_xlabel('Lift')
ax.set_title('Top 15 asocijacijskih pravila po Lift-u')
ax.legend()
plt.tight_layout()
plt.show()

# Lift distribucija
fig, ax = plt.subplots(figsize=(10, 4))
rules['lift'].hist(bins=40, ax=ax, color='steelblue', edgecolor='black')
ax.axvline(x=1.0, color='red', linestyle='--', label='Lift=1')
ax.set_xlabel('Lift')
ax.set_title('Distribucija Lift vrijednosti')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 10: Co-occurrence heatmapa
# ============================================================
# Vizualiziramo prosječni lift između svih parova PG.
# Dijagonala je uvijek 1 (PG sa samim sobom).

# Uzmi samo pravila s jednim antecedentom i jednim konsekventom
pair_rules = rules[
    (rules['antecedents'].apply(len) == 1) &
    (rules['consequents'].apply(len) == 1)
].copy()

pair_rules['ant'] = pair_rules['antecedents'].apply(lambda x: list(x)[0])
pair_rules['con'] = pair_rules['consequents'].apply(lambda x: list(x)[0])

# Pivot u matricu
lift_matrix = pair_rules.pivot_table(
    index='ant', columns='con', values='lift', aggfunc='mean'
)

# Ograniči na top PG ako ih ima previše
if lift_matrix.shape[0] > 20:
    top_pg = pg_counts.head(20).index.tolist()
    lift_matrix = lift_matrix.loc[
        lift_matrix.index.isin(top_pg),
        lift_matrix.columns.isin(top_pg)
    ]

if not lift_matrix.empty:
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(
        lift_matrix,
        annot=True, fmt='.2f',
        cmap='YlOrRd',
        center=1.0,
        linewidths=0.5,
        ax=ax
    )
    ax.set_title('Lift heatmapa — Product Group parovi (>1 = pozitivna asocijacija)')
    plt.tight_layout()
    plt.show()
else:
    print('Nema dovoljno parnih pravila za heatmapu — smanji min_support ili min_confidence.')

In [ ]:
# ============================================================
# CELL 11: Per-customer cross-sell preporuke
# ============================================================
# Logika:
#   1. Za svakog kupca: koje Product Group je ikada kupio?
#   2. Traži sva pravila gdje je antecedent ⊆ kupčeve PG
#   3. Consequent koji kupac NIJE kupio = cross-sell kandidat
#   4. Rangiraj po lift-u

# Kupčev portfelj PG
customer_pg = (
    df.groupby('CustomerID')['Product group']
    .apply(lambda x: frozenset(x.dropna().unique()))
    .reset_index()
)
customer_pg.columns = ['CustomerID', 'bought_pg']

print(f'Kupaca za obradu: {len(customer_pg):,}')
print(f'Pravila za pretragu: {len(rules):,}')

# Pripremi pravila za brzi lookup
rules_lookup = rules[['antecedents', 'consequents', 'lift', 'confidence']].copy()

def get_recommendations(bought_set, rules_df, top_n=5):
    """
    Za dani skup kupljenih PG, vrati top_n cross-sell preporuka.
    Pravilo se primjenjuje ako su SVI antecedenti u bought_set
    a konsekventa NEMA u bought_set.
    """
    recs = []
    for _, row in rules_df.iterrows():
        if row['antecedents'].issubset(bought_set):
            new_items = row['consequents'] - bought_set
            for item in new_items:
                recs.append({
                    'recommended_pg': item,
                    'lift': row['lift'],
                    'confidence': row['confidence']
                })
    if not recs:
        return []
    # Dedupliciraj — za isti PG zadrži pravilo s najvišim lift-om
    recs_df = pd.DataFrame(recs).sort_values('lift', ascending=False)
    recs_df = recs_df.drop_duplicates(subset='recommended_pg')
    return recs_df.head(top_n).to_dict('records')

# Primjer za prvih 5 kupaca
print('\n=== Primjer preporuka za 5 kupaca ===')
for _, row in customer_pg.head(20).iterrows():
    recs = get_recommendations(row['bought_pg'], rules_lookup, top_n=3)
    print(f'\nKupac {row["CustomerID"]} (kupuje {len(row["bought_pg"])} PG):')
    if recs:
        for r in recs:
            print(f'  → {r["recommended_pg"]:15s} | lift={r["lift"]:.3f} | confidence={r["confidence"]:.3f}')
    else:
        print('  (nema preporuka — kupuje sve ili nema matching pravila)')

# samo demo ovog dolje, ali je out of scope

In [ ]:
"""
# ============================================================
# CELL 12: Generiraj preporuke za SVE kupce
# ============================================================
# Vektoriziraniji pristup: pretvaramo pravila u listu tuplova
# za brži lookup umjesto iterrows na rules_df u svakom pozivu.

print('Generiranje preporuka za sve kupce...')

# Pretvori pravila u listu tuplova za brži pristup
rules_tuples = [
    (row['antecedents'], row['consequents'], row['lift'], row['confidence'])
    for _, row in rules_lookup.iterrows()
]

def get_recs_fast(bought_set, rules_tuples, top_n=5):
    """Brža verzija — iterira po listi tuplova umjesto DataFrame."""
    recs = {}
    for ant, con, lift, conf in rules_tuples:
        if ant.issubset(bought_set):
            for item in con - bought_set:
                # Zadrži samo pravilo s najvišim liftom za svaki PG
                if item not in recs or lift > recs[item][0]:
                    recs[item] = (lift, conf)
    # Sortiraj po liftu i vrati top_n
    sorted_recs = sorted(recs.items(), key=lambda x: x[1][0], reverse=True)[:top_n]
    return [(pg, lift, conf) for pg, (lift, conf) in sorted_recs]

t0 = time.time()
all_recs = []

for _, row in customer_pg.iterrows():
    recs = get_recs_fast(row['bought_pg'], rules_tuples, top_n=5)
    for pg, lift, conf in recs:
        all_recs.append({
            'CustomerID': row['CustomerID'],
            'recommended_pg': pg,
            'lift': lift,
            'confidence': conf,
            'type': 'cross-sell'
        })

t1 = time.time()
customer_crosssell = pd.DataFrame(all_recs)

print(f'Gotovo za {t1-t0:.1f}s')
print(f'Generirano preporuka: {len(customer_crosssell):,}')
print(f'Kupaca s barem jednom preporukom: {customer_crosssell["CustomerID"].nunique():,}')
print(f'Kupaca BEZ preporuka: {len(customer_pg) - customer_crosssell["CustomerID"].nunique():,}')

print('\nPrimjer outputa:')
display(customer_crosssell.head(10))



# ============================================================
# CELL 13: Analiza pokrivenosti preporuka
# ============================================================
# Provjeri kvalitetu preporuka — distribucija liftova,
# koliko preporuka po kupcu, koji PG se najčešće preporučuje.

if len(customer_crosssell) > 0:
    recs_per_customer = customer_crosssell.groupby('CustomerID').size()
    print('=== Preporuka po kupcu ===')
    print(recs_per_customer.describe())

    print(f'\nNajčešće preporučeni PG:')
    print(customer_crosssell['recommended_pg'].value_counts().head(10))

    print(f'\nDistribucija lift-a preporuka:')
    print(customer_crosssell['lift'].describe())

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    recs_per_customer.clip(upper=5).value_counts().sort_index().plot(
        kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
    axes[0].set_title('Broj preporuka po kupcu')
    axes[0].set_xlabel('Broj preporuka')

    customer_crosssell['lift'].hist(bins=30, ax=axes[1], color='steelblue', edgecolor='black')
    axes[1].set_title('Distribucija lift-a preporuka')
    axes[1].axvline(x=1.0, color='red', linestyle='--', label='Lift=1')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print('Nema preporuka — provjeri min_support i min_confidence parametre.')
"""
# necemo ovo radit jer je out of scope, ali bilo bi korisno za marketing kampanje
# tu bi se proslo kroz sve kupce i temeljem pravila i onog cega su vec kupovali vidjelo dal se neko pravilo moze primjeniti na njih
# i poslati neke personalizirane mailove gdje im se preporucuje taj product group koji jos nisu kupovali

In [ ]:
# ============================================================
# CELL 14: Spremi outpute
# ============================================================
# Tri outputa:
#   1. association_rules.csv — sva pravila s metrikama
#   2. customer_crosssell.csv — cross-sell po kupcu
#   3. basket_matrix.csv — one-hot matrica za ML model

# Pravila — spremi string verzije + metrike (frozenset se ne može u CSV)
rules_export = rules[[
    'antecedents_str', 'consequents_str',
    'support', 'confidence', 'lift', 'leverage', 'conviction'
]].copy()
rules_export.to_csv('../data/processed/association_rules.csv', index=False)
print(f'Spremljeno: association_rules.csv ({len(rules_export)} pravila)')

"""
customer_crosssell.to_csv('../data/processed/customer_crosssell.csv', index=False)
print(f'Spremljeno: customer_crosssell.csv ({len(customer_crosssell)} redova)')
"""

basket_matrix.to_csv('../data/processed/basket_matrix.csv', index=False)
print(f'Spremljeno: basket_matrix.csv ({basket_matrix.shape})')

print('\n✓ Market basket analysis gotova!')

In [ ]:
# ============================================================
# BONUS: Item Code MBA — čisti pandas pristup, bez mlxtend
# ============================================================

import pandas as pd
import numpy as np
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

MIN_ITEM_COUNT = 500   # samo 112 itemova → sigurno
MIN_SUPPORT    = 0.002 # par mora biti u 2‰ basketa
MIN_CONFIDENCE = 0.05
MIN_LIFT       = 1.0

# --- Filter frequent itemova ---
item_inv = df.groupby('Item Code')['Invoice #'].nunique()
freq_items = item_inv[item_inv >= MIN_ITEM_COUNT].index
print(f"Frequent itemova (>= {MIN_ITEM_COUNT} faktura): {len(freq_items)}")

df_freq = df[df['Item Code'].isin(freq_items)].copy()

# --- Basket definicija ---
df_freq['basket_id'] = (
    df_freq['CustomerID'].astype(str) + '_' +
    df_freq['Invoice Date'].dt.to_period('M').astype(str)
)

# Deduplikacija: jedan item jednom po basketu
item_basket = (
    df_freq[['basket_id', 'Item Code']]
    .drop_duplicates()
)

N_BASKETS = item_basket['basket_id'].nunique()
print(f"Ukupno basketa: {N_BASKETS:,}")

# --- Individualni support po itemu ---
item_support = (
    item_basket.groupby('Item Code')['basket_id']
    .nunique()
    .rename('item_count')
)

# --- Pairwise co-occurrence via self-join ---
print("Računam pairwise co-occurrences (self-join)...")

merged = item_basket.merge(item_basket, on='basket_id')
# Samo jedan smjer para (A < B) da izbjegnemo duplikate
merged = merged[merged['Item Code_x'] < merged['Item Code_y']]

pair_counts = (
    merged
    .groupby(['Item Code_x', 'Item Code_y'])
    .size()
    .reset_index(name='co_count')
)

print(f"Unikatnih parova pronađeno: {len(pair_counts):,}")

# --- Izračun metrika ---
pair_counts['support']    = pair_counts['co_count'] / N_BASKETS
pair_counts['support_A']  = pair_counts['Item Code_x'].map(item_support) / N_BASKETS
pair_counts['support_B']  = pair_counts['Item Code_y'].map(item_support) / N_BASKETS
pair_counts['conf_A→B']   = pair_counts['support'] / pair_counts['support_A']
pair_counts['conf_B→A']   = pair_counts['support'] / pair_counts['support_B']
pair_counts['lift']       = pair_counts['support'] / (pair_counts['support_A'] * pair_counts['support_B'])

# --- Filtriraj po pragovima ---
rules = pair_counts[
    (pair_counts['support']  >= MIN_SUPPORT) &
    (pair_counts['lift']     >= MIN_LIFT)    &
    (pair_counts[['conf_A→B','conf_B→A']].max(axis=1) >= MIN_CONFIDENCE)
].sort_values('lift', ascending=False)

print(f"Pravila (support>={MIN_SUPPORT}, lift>={MIN_LIFT}, conf>={MIN_CONFIDENCE}): {len(rules):,}")
print()

if len(rules) > 0:
    print("TOP 20 PAROVA PO LIFTU:")
    print("-" * 80)
    top = rules.head(20)[['Item Code_x','Item Code_y',
                           'support','conf_A→B','conf_B→A','lift']].copy()
    top['support']   = top['support'].map('{:.4f}'.format)
    top['conf_A→B']  = top['conf_A→B'].map('{:.3f}'.format)
    top['conf_B→A']  = top['conf_B→A'].map('{:.3f}'.format)
    top['lift']      = top['lift'].map('{:.2f}'.format)
    print(top.to_string(index=False))
    print(f"\nLift: min={rules['lift'].min():.2f}  "
          f"median={rules['lift'].median():.2f}  "
          f"max={rules['lift'].max():.2f}")
else:
    print("⚠️  Nema parova — čak i najpopularnijih 112 itemova se rijetko kupuje zajedno.")
    print(f"   Proba: item_inv.nlargest(5):")
    print(f"   {item_inv.nlargest(5).to_dict()}")

# --- ZAKLJUČAK ---
print("""
╔══════════════════════════════════════════════════════════════╗
║  ZAKLJUČAK: Item Code MBA                                    ║
╠══════════════════════════════════════════════════════════════╣
║  Čak top 112 itemova (po frekvenciji) generira minimalne     ║
║  asocijacije — B2B purchasing pattern je previše             ║
║  idiosinkratičan po kupcu za globalnu item-level analizu.    ║
║  Product Group ostaje jedini praktičan pristup.              ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## Sažetak — što smo napravili

| Korak | Akcija |
|-------|--------|
| EDA | Analiza Product group distribucije i složenosti faktura |
| Basket priprema | TransactionEncoder → boolean matrica (Invoice × PG) |
| Apriori | Klasični algoritam s antimonotonosti — objašnjenje teorije |
| FP-Growth | Brži drop-in replacement — isti rezultati, koristimo dalje |
| Pravila | Filtrirana po lift>1.0 i confidence>0.2, uključujući conviction i leverage |
| Vizualizacija | Scatter plot, bar chart top pravila, co-occurrence heatmapa |
| Cross-sell | Per-customer preporuke iz asocijacijskih pravila |
| Output | 3 CSV-a: pravila, preporuke, basket matrica |

> **Napomena:** Up-sell (kupci ispod medijana cijene po PG) nije dio ovog notebooka
> jer ne koristi co-occurrence podatke. To se implementira u recommendation modelu / API-ju.